## Importing Libraries & Setting Working Directory

In [2]:
import os

# Set working directory to the Analysis folder (optional, usually automatic)
os.chdir("/Users/ghazalhashmi/Library/CloudStorage/Box-Box/Ghazal/Trial")


In [3]:
#Confirm current working directory
print("Current working directory:", os.getcwd())

Current working directory: /Users/ghazalhashmi/Library/CloudStorage/Box-Box/Ghazal/Trial


## Connecting to SQLite Database & Loading Data

In [4]:
import sqlite3
import pandas as pd

# Relative path from Analysis → Data/Raw
db_path = "/Users/ghazalhashmi/Library/CloudStorage/Box-Box/Ghazal/Data/Raw/grievance_complaints.db"

In [5]:
# Connect
conn = sqlite3.connect(db_path)

In [6]:
# See all tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
tables

,name
0,complaints


In [7]:
import numpy as np
import pandas as pd

In [8]:
# Load all rows from the complaints table
df = pd.read_sql_query("SELECT * FROM complaints;", conn)

## Creating a Working Copy

In [9]:
df_work = df.copy()    # safe copy; raw df remains untouched
print("Raw shape:", df.shape)
print("Working copy shape:", df_work.shape)


Raw shape: (1371288, 50)
Working copy shape: (1371288, 50)


In [10]:
df_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371288 entries, 0 to 1371287
Data columns (total 50 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   id                       1371288 non-null  int64  
 1   ticket_no                1371288 non-null  object 
 2   petitioner_name          1371174 non-null  object 
 3   petitioner_mobile        1370545 non-null  object 
 4   petitioner_email         1356047 non-null  object 
 5   grievance                1371285 non-null  object 
 6   document_url             1047100 non-null  object 
 7   office_id                1371288 non-null  int64  
 8   office                   1311838 non-null  object 
 9   received_by              1371278 non-null  object 
 10  district_id              1366678 non-null  float64
 11  district                 1361869 non-null  object 
 12  block_id                 1371288 non-null  int64  
 13  block                    1134155 non-null 

In [11]:
df_work.columns

Index(['id', 'ticket_no', 'petitioner_name', 'petitioner_mobile',
       'petitioner_email', 'grievance', 'document_url', 'office_id', 'office',
       'received_by', 'district_id', 'district', 'block_id', 'block',
       'address', 'mode', 'disability', 'status', 'govt_ticket', 'created_on',
       'tagged_to', 'tagged_by', 'tagged_date', 'category_id', 'category',
       'dept_id', 'dept', 'subcategory_id', 'subcategory', 'state',
       'petitioner_gender', 'transfer_status', 'urgent', 'pending_with',
       'assigned_on', 'escalation_date', 'self_assign', 'resolved_by',
       'resolved_on', 'benefitted', 'local_document_path',
       'document_downloaded', 'document_download_date',
       'document_download_error', 'trackingId', 'review_authority',
       'review_authority_name', 'vch_all_esc_user', 'reopened_by',
       'vch_account'],
      dtype='object')

In [12]:
df_work.head()

,id,ticket_no,petitioner_name,petitioner_mobile,petitioner_email,grievance,document_url,office_id,office,received_by,...,local_document_path,document_downloaded,document_download_date,document_download_error,trackingId,review_authority,review_authority_name,vch_all_esc_user,reopened_by,vch_account
0,1,C132910,Subhabr49917722,******rata,,&lt;a href=&quot;http://www.twitter.com/CMCCut...,None,0,None,Chief Minister Office,...,None,0,None,None,1377613503155036161,81,Chief Minister Office,"73,44,81",None,MENTION_TW
1,2,C143339,JasobantaRana2,******nta,,Respected &lt;a href=&quot;http://www.twitter....,None,0,None,Chief Minister Office,...,None,0,None,None,1388252153962065922,81,Chief Minister Office,"2,54,81",None,MENTION_TW
2,3,C149351,Bishwaksenkuanr,******ksen,,&lt;a href=&quot;http://www.twitter.com/CMO_Od...,None,0,None,Chief Minister Office,...,None,0,None,None,1393753306774728706,81,Chief Minister Office,"7,54,81",None,MENTION_TW
3,4,C150702,frozenfire_aj,******Kuma,,&lt;a href=&quot;http://www.twitter.com/SonuSo...,None,0,None,Chief Minister Office,...,None,0,None,None,1394890123616030721,81,Chief Minister Office,"12,43,81",None,MENTION_TW
4,5,C165110,woymforpeople,******n Od,,"Gita Naik, A widow alongwith 3 daughter living...",None,0,None,Chief Minister Office,...,None,0,None,None,1407958304760819713,81,Chief Minister Office,"16,62,81",None,MENTION_TW


In [13]:
# Missing value summary
missing_summary = (
    df_work.isnull()
    .sum()
    .reset_index()
    .rename(columns={'index': 'Column', 0: 'Missing_Values'})
)
missing_summary["Missing_%"] = round((missing_summary["Missing_Values"] / len(df_work)) * 100, 2)

# Sort by most missing
missing_summary = missing_summary.sort_values("Missing_%", ascending=False)
missing_summary


,Column,Missing_Values,Missing_%
43,document_download_error,1371288,100.00
40,local_document_path,1371288,100.00
42,document_download_date,1371288,100.00
22,tagged_date,1358678,99.08
21,tagged_by,1358673,99.08
20,tagged_to,1337012,97.50
48,reopened_by,1269523,92.58
16,disability,552459,40.29
6,document_url,324188,23.64
13,block,237133,17.29


## Selecting Columns for Analysis

In [14]:
analysis_cols = [
    'id', 'ticket_no', 'document_url',
    'office_id', 'office', 'received_by',
    'district_id', 'district', 'mode', 'disability', 'status', 'created_on',
    'category_id', 'category', 'dept_id', 'dept',
    'subcategory_id', 'subcategory', 'state', 'petitioner_gender'
]

df_work = df_work[analysis_cols].copy()
print("Shape:", df_work.shape)


Shape: (1371288, 20)


In [15]:
df_work.info()
df_work.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371288 entries, 0 to 1371287
Data columns (total 20 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   id                 1371288 non-null  int64  
 1   ticket_no          1371288 non-null  object 
 2   document_url       1047100 non-null  object 
 3   office_id          1371288 non-null  int64  
 4   office             1311838 non-null  object 
 5   received_by        1371278 non-null  object 
 6   district_id        1366678 non-null  float64
 7   district           1361869 non-null  object 
 8   mode               1371288 non-null  object 
 9   disability         818829 non-null   object 
 10  status             1371288 non-null  object 
 11  created_on         1371288 non-null  object 
 12  category_id        1371288 non-null  int64  
 13  category           1139836 non-null  object 
 14  dept_id            1371288 non-null  int64  
 15  dept               1160838 non-n

Index(['id', 'ticket_no', 'document_url', 'office_id', 'office', 'received_by',
       'district_id', 'district', 'mode', 'disability', 'status', 'created_on',
       'category_id', 'category', 'dept_id', 'dept', 'subcategory_id',
       'subcategory', 'state', 'petitioner_gender'],
      dtype='object')

In [16]:
df_work.head()

,id,ticket_no,document_url,office_id,office,received_by,district_id,district,mode,disability,status,created_on,category_id,category,dept_id,dept,subcategory_id,subcategory,state,petitioner_gender
0,1,C132910,None,0,None,Chief Minister Office,350.0,Cuttack,Twitter,None,Disposed,2021-04-01 19:09:34.000000,10,Infrastructure,15,Housing & Urban Development,51,Road,ODISHA,Other
1,2,C143339,None,0,None,Chief Minister Office,344.0,Angul,Twitter,None,Disposed,2021-05-01 10:39:47.000000,39,Water Supply,21,Panchayati Raj & Drinking Water,171,Tubewell/Borewell (Drinking Water),ODISHA,Other
2,3,C149351,None,0,None,Chief Minister Office,349.0,Boudh,Twitter,None,Disposed,2021-05-16 12:11:30.000000,39,Water Supply,21,Panchayati Raj & Drinking Water,171,Tubewell/Borewell (Drinking Water),ODISHA,Other
3,4,C150702,None,0,None,Chief Minister Office,362.0,Khordha,Twitter,None,Disposed,2021-05-19 14:30:32.000000,8,Health Care,12,Health & Family Welfare,169,Treatment Assistance,ODISHA,Other
4,5,C165110,None,0,None,Chief Minister Office,NaN,None,Twitter,None,Disposed,2021-06-24 13:59:04.000000,7,Social Welfare,40,Social Security & Empowerment of Persons with ...,35,Social Security Scheme/ MBPY,ODISHA,Other


In [17]:
df_work.isnull().mean().sort_values(ascending=False) * 100


disability           40.287598
document_url         23.641132
category             16.878438
dept                 15.346886
subcategory          15.089755
office                4.335340
district              0.686872
district_id           0.336180
state                 0.014585
received_by           0.000729
id                    0.000000
subcategory_id        0.000000
dept_id               0.000000
status                0.000000
category_id           0.000000
created_on            0.000000
ticket_no             0.000000
mode                  0.000000
office_id             0.000000
petitioner_gender     0.000000
dtype: float64

## Disability Column Checks

In [18]:
df_work['disability'].value_counts(dropna=False)


disability
None                                    769149
None                                    552459
Other Disability                         29210
Locomotor Disability /Cerebral Palsy     12301
Blindness / Low Vision                    4858
Mental Illness                            3311
Name: count, dtype: int64

In [19]:
df_work['disability'].unique()

array([None, 'None', 'Locomotor Disability /Cerebral Palsy',
       'Blindness / Low Vision', 'Other Disability', 'Mental Illness'],
      dtype=object)

In [20]:
df_work['disability'].value_counts().sum()


np.int64(818829)

In [21]:
df_work['disability'].apply(type).value_counts()


disability
<class 'str'>         818829
<class 'NoneType'>    552459
Name: count, dtype: int64

### Cross Tab Analysis for NaN and 'None' Disability Categories

To understand whether missing disability information (NaN values) occurs more frequently during online mode submissions (e.g., Website, Whatsapp) or offline submissions (e.g., Physical, Letter, CMO visits).

In [ ]:
#Generate Counts for All Disability Categories
pd.crosstab(df_work['disability'], df_work['mode'], dropna=False)


mode,CM Weekly Grievance,CMO District Visit I - Community Grievance,CMO District Visit I - Individual Grievance,CMO District Visit II - Community Grievance,CMO District Visit II - Individual Grievance,Email,Facebook,Joint Hearing,Letter,Mobile,Physical,Twitter,Urgent,Website,Whatsapp
disability,,,,,,,,,,,,,,,
Blindness / Low Vision,25,2,4,3,11,8,0,94,62,174,385,0,0,3456,634
Locomotor Disability /Cerebral Palsy,96,5,17,31,49,43,0,529,341,1174,1109,0,0,7544,1363
Mental Illness,8,0,4,5,7,7,0,67,33,351,130,0,0,2215,484
None,11702,7610,1478,27381,6696,8574,0,49893,39612,19293,88504,0,333,470318,37755
Other Disability,145,31,26,261,145,22,0,635,423,2649,1093,0,3,17178,6599
NaN,43,36579,10647,152,91,26826,47,422,25451,20069,68781,59465,2,243475,60409


In [ ]:
#Generate Percentage for All Disability Categories
ct = pd.crosstab(
    df_work['disability'],
    df_work['mode'],
    dropna=False,
    normalize='index'
) * 100
ct


mode,CM Weekly Grievance,CMO District Visit I - Community Grievance,CMO District Visit I - Individual Grievance,CMO District Visit II - Community Grievance,CMO District Visit II - Individual Grievance,Email,Facebook,Joint Hearing,Letter,Mobile,Physical,Twitter,Urgent,Website,Whatsapp
disability,,,,,,,,,,,,,,,
Blindness / Low Vision,0.514615,0.041169,0.082338,0.061754,0.226431,0.164677,0.000000,1.934953,1.276245,3.581721,7.925072,0.000000,0.000000,71.140387,13.050638
Locomotor Disability /Cerebral Palsy,0.780424,0.040647,0.138200,0.252012,0.398342,0.349565,0.000000,4.300463,2.772132,9.543940,9.015527,0.000000,0.000000,61.328347,11.080400
Mental Illness,0.241619,0.000000,0.120809,0.151012,0.211416,0.211416,0.000000,2.023558,0.996678,10.601027,3.926306,0.000000,0.000000,66.898218,14.617940
None,1.521422,0.989405,0.192160,3.559908,0.870573,1.114738,0.000000,6.486780,5.150107,2.508357,11.506743,0.000000,0.043295,61.147840,4.908672
Other Disability,0.496405,0.106128,0.089011,0.893530,0.496405,0.075317,0.000000,2.173913,1.448134,9.068812,3.741869,0.000000,0.010270,58.808627,22.591578
NaN,0.007783,6.621125,1.927202,0.027513,0.016472,4.855745,0.008507,0.076386,4.606858,3.632668,12.449974,10.763695,0.000362,44.071144,10.934567


In [ ]:
#Filter Crosstab for Only NaN and "None"
ct.loc[ct.index.isna() | (ct.index == "None")]


mode,CM Weekly Grievance,CMO District Visit I - Community Grievance,CMO District Visit I - Individual Grievance,CMO District Visit II - Community Grievance,CMO District Visit II - Individual Grievance,Email,Facebook,Joint Hearing,Letter,Mobile,Physical,Twitter,Urgent,Website,Whatsapp
disability,,,,,,,,,,,,,,,
None,1.521422,0.989405,0.192160,3.559908,0.870573,1.114738,0.000000,6.486780,5.150107,2.508357,11.506743,0.000000,0.043295,61.147840,4.908672
NaN,0.007783,6.621125,1.927202,0.027513,0.016472,4.855745,0.008507,0.076386,4.606858,3.632668,12.449974,10.763695,0.000362,44.071144,10.934567


> **Note:**  
> Standardization steps are commented out, so all original values (including `NaN` and `None`) were kept unchanged.


In [ ]:
#Make NaN and 'None' Consistent
# #df_work['disability'] = (
    #df_work['disability']
    #.astype(str)                     
    #.str.strip()                    
    #.replace(
        #['None', 'none', 'NONE', 'nan', 'NaN', 'NULL', 'null', '', ' ', 'Not Available', 'Not Applicable'],'Not Mentioned)
    #.fillna('Not Mentioned'))


In [ ]:
#df_work['disability'].value_counts(dropna=False)

disability
Not Mentioned                           1321608
Other Disability                          29210
Locomotor Disability /Cerebral Palsy      12301
Blindness / Low Vision                     4858
Mental Illness                             3311
Name: count, dtype: int64

## Petitioner Gender Column Check

In [ ]:
df_work['petitioner_gender'].value_counts(dropna=False)

petitioner_gender
Male           610008
Female         463562
Other          296903
Transgender       815
Name: count, dtype: int64

## Ticket_No Column Check

In [ ]:
df_work['ticket_no'].value_counts(dropna=False)

ticket_no
C132910           1
DM2023234693      1
DEPT2023230552    1
DEPT2023231265    1
CMO2023230162     1
                 ..
SP202293213       1
GOV202294692      1
CMO202296914      1
DM202294696       1
DM20251286400     1
Name: count, Length: 1371288, dtype: int64

In [ ]:
df_work[df_work['ticket_no'].duplicated(keep=False)].sort_values('ticket_no')


,id,ticket_no,document_url,office_id,office,received_by,district_id,district,mode,disability,status,created_on,category_id,category,dept_id,dept,subcategory_id,subcategory,state,petitioner_gender


In [ ]:
df_work['ticket_no'].nunique(), len(df_work)


(1371288, 1371288)

In [ ]:
#length of ticket numbers
df_work['ticket_no'].astype(str).str.len().value_counts().sort_index()

ticket_no
7      58939
11     57931
12    599037
13    307192
14    252734
15     67433
17       338
18     27684
Name: count, dtype: int64

In [ ]:
# Loop through each unique ticket length (calculated on the fly)
for length in sorted(df_work['ticket_no'].astype(str).str.len().unique()):
    print(f"\nTicket length = {length}")
    # Show first 5 examples of tickets having that length
    print(df_work.loc[
        df_work['ticket_no'].astype(str).str.len() == length, 'ticket_no'
    ].head(5).to_list())



Ticket length = 7
['C132910', 'C143339', 'C149351', 'C150702', 'C165110']

Ticket length = 11
['DM202108451', 'SP202112653', 'DM202114686', 'SP202219079', 'DM202220158']

Ticket length = 12
['CMO202100019', 'CMO202102387', 'CMO202103782', 'CMO202218570', 'CMO202225613']

Ticket length = 13
['DEPT202101180', 'DEPT202226527', 'DEPT202234869', 'DEPT202236643', 'DEPT202237768']

Ticket length = 14
['DEPT2022104524', 'DEPT2022108498', 'DEPT2022118481', 'DEPT2022120916', 'DEPT2022130173']

Ticket length = 15
['DEPT20241010126', 'DEPT20241026968', 'DEPT20241025428', 'DEPT20241023573', 'DEPT20241044383']

Ticket length = 17
['URGENT20251232180', 'URGENT20251235997', 'URGENT20251248139', 'URGENT20251235751', 'URGENT20251239031']

Ticket length = 18
['CMOFF/E/2021/10899', 'OR149/E/2021/00593', 'CMOFF/E/2021/18868', 'OR180/D/2021/01178', 'OR171/P/2021/00429']


### Ticket Length vs. Created Date Summary

To check how the length of the ticket number relates to the time period during which the complaints were created. This helps identify changes in ticket formats or system updates over time.

In [ ]:
#Created On (Date-Time) Processing
df_work['created_on'] = pd.to_datetime(df_work['created_on'], errors='coerce')


In [ ]:
#Earliest and latest complaint dates for each length
(
    df_work
    .assign(ticket_length=df_work['ticket_no'].astype(str).str.len()) 
    .groupby('ticket_length')['created_on']
    .agg(['min', 'max'])
    .sort_index()
)


,min,max
ticket_length,,
7,2021-04-01 10:17:02,2025-07-29 17:58:22
11,2021-10-30 21:34:49,2022-09-25 17:59:58
12,2021-10-30 22:25:28,2024-11-29 11:17:54
13,2021-11-01 07:57:33,2025-07-30 11:34:52
14,2022-09-25 18:04:53,2025-07-30 11:31:39
15,2024-11-29 11:20:59,2025-07-30 11:34:18
17,2025-05-27 13:23:25,2025-07-10 16:39:39
18,2021-04-05 00:00:00,2021-10-31 00:00:00


## ID Column Check

In [ ]:
df_work['id'].value_counts(dropna=False)

id
1          1
914199     1
914197     1
914196     1
914195     1
          ..
457096     1
457095     1
457094     1
457093     1
1371288    1
Name: count, Length: 1371288, dtype: int64

In [ ]:
df_work['id'].tail(10)

1371278    1371279
1371279    1371280
1371280    1371281
1371281    1371282
1371282    1371283
1371283    1371284
1371284    1371285
1371285    1371286
1371286    1371287
1371287    1371288
Name: id, dtype: int64

In [ ]:
df_work['id'].unique()


array([      1,       2,       3, ..., 1371286, 1371287, 1371288])

## Document_url Column Check

In [ ]:
df_work['document_url'].value_counts(dropna=False)


document_url
None                                                                                                                          324188
https://janasunani.odisha.gov.in/getfileApi/uploads/ecomplaint/2022EA/CRM_null__COGNIZANCE~pdf                                    40
https://janasunani.odisha.gov.in/getfileApi/uploads/ecomplaint/2022EA/CRM_null__83414~pdf                                         11
https://janasunani.odisha.gov.in/getfileApi/uploads/ecomplaint/2022EA/CRM_null__DOC-20220916-WA0024~~pdf                           7
https://janasunani.odisha.gov.in/getfileApi/uploads/ecomplaint/2022EA/CRM_null__chitra ranjan~pdf                                  7
                                                                                                                               ...  
https://janasunani.odisha.gov.in/getfileApi/uploads/ecomplaint/2025EA/abhijogaDoc20250129_0537161738129036784_9823~pdf             1
https://janasunani.odisha.gov.in/getfileApi/uploads/ecom

In [ ]:
# Create binary variable Yes/No for document attached
df_work['document_attached'] = df_work['document_url'].notna()
df_work['document_attached'] = df_work['document_attached'].replace({True: 'Yes', False: 'No'})
df_work['document_attached'].value_counts(normalize=True) * 100

document_attached
Yes    76.358868
No     23.641132
Name: proportion, dtype: float64

In [ ]:
df_work[['document_url', 'document_attached']].head(10)

,document_url,document_attached
0,None,No
1,None,No
2,None,No
3,None,No
4,None,No
5,None,No
6,None,No
7,https://janasunani.odisha.gov.in/getfileApi/up...,Yes
8,https://janasunani.odisha.gov.in/getfileApi/up...,Yes
9,https://janasunani.odisha.gov.in/getfileApi/up...,Yes


### Mode vs Document Attachment Crosstab

To compare each complaint submission mode with whether a supporting document was attached. It helps identify which modes (online or offline) tend to include attachments more often.

In [ ]:
pd.crosstab(
    df_work['mode'],
    df_work['document_attached'],
    margins=True,
    normalize='index'
)


document_attached,No,Yes
mode,,
CM Weekly Grievance,0.004493,0.995507
CMO District Visit I - Community Grievance,0.145341,0.854659
CMO District Visit I - Individual Grievance,0.173456,0.826544
CMO District Visit II - Community Grievance,0.002910,0.997090
CMO District Visit II - Individual Grievance,0.002143,0.997857
Email,0.187486,0.812514
Facebook,1.000000,0.000000
Joint Hearing,0.003776,0.996224
Letter,0.011028,0.988972


## Mode Column Check

In [ ]:
df_work['mode'].value_counts(dropna=False)

mode
Website                                         744186
Physical                                        160002
Whatsapp                                        107244
Letter                                           65922
Twitter                                          59465
Joint Hearing                                    51640
CMO District Visit I - Community Grievance       44227
Mobile                                           43710
Email                                            35480
CMO District Visit II - Community Grievance      27833
CMO District Visit I - Individual Grievance      12176
CM Weekly Grievance                              12019
CMO District Visit II - Individual Grievance      6999
Urgent                                             338
Facebook                                            47
Name: count, dtype: int64

In [ ]:
df_work['mode'].isnull().sum()

np.int64(0)

## Status Column Check

In [ ]:
df_work['status'].value_counts(dropna=False)

status
Disposed                    1017795
Discard                      191264
Open                          52130
Replied                       42095
Forwarded To Subordinate      31731
Not Assigned                  26378
Reopen                         4113
Forward                        3675
RESOLVED_POSTING               1750
Accepted                        239
In Follow Up With Close          85
In Follow Up With Open           33
Name: count, dtype: int64

In [ ]:
df_work['status'].isnull().sum()

np.int64(0)

## Created_On Column Check

In [ ]:
df_work['created_on'].head(10)


0   2021-04-01 19:09:34
1   2021-05-01 10:39:47
2   2021-05-16 12:11:30
3   2021-05-19 14:30:32
4   2021-06-24 13:59:04
5   2021-07-23 20:10:39
6   2021-09-09 11:20:57
7   2021-11-01 09:35:18
8   2021-11-09 09:00:34
9   2021-11-15 13:35:06
Name: created_on, dtype: datetime64[ns]

In [ ]:
df_work['created_on'].isnull().sum()

np.int64(0)

In [ ]:
df_work['created_on'] = pd.to_datetime(df_work['created_on'])


In [ ]:
#Range of Created_On dates
df_work['created_on'].agg(['min', 'max'])


min   2021-04-01 10:17:02
max   2025-07-30 11:34:52
Name: created_on, dtype: datetime64[ns]

In [ ]:
#Add time-based columns
df_work['year'] = df_work['created_on'].dt.year
df_work['month'] = df_work['created_on'].dt.month
df_work['month_name'] = df_work['created_on'].dt.month_name()
df_work['day'] = df_work['created_on'].dt.day
df_work['day_of_week'] = df_work['created_on'].dt.day_name()
df_work['hour'] = df_work['created_on'].dt.hour
df_work['date'] = df_work['created_on'].dt.date

In [ ]:
df_work[['created_on', 'year', 'month_name', 'day', 'day_of_week', 'hour', 'date']].head(10)

,created_on,year,month_name,day,day_of_week,hour,date
0,2021-04-01 19:09:34,2021,April,1,Thursday,19,2021-04-01
1,2021-05-01 10:39:47,2021,May,1,Saturday,10,2021-05-01
2,2021-05-16 12:11:30,2021,May,16,Sunday,12,2021-05-16
3,2021-05-19 14:30:32,2021,May,19,Wednesday,14,2021-05-19
4,2021-06-24 13:59:04,2021,June,24,Thursday,13,2021-06-24
5,2021-07-23 20:10:39,2021,July,23,Friday,20,2021-07-23
6,2021-09-09 11:20:57,2021,September,9,Thursday,11,2021-09-09
7,2021-11-01 09:35:18,2021,November,1,Monday,9,2021-11-01
8,2021-11-09 09:00:34,2021,November,9,Tuesday,9,2021-11-09
9,2021-11-15 13:35:06,2021,November,15,Monday,13,2021-11-15


## State Column Check

In [ ]:
df_work['state'].value_counts(dropna=False)

state
ODISHA                         1366623
DELHI                             1032
MADHYA PRADESH                     881
WEST BENGAL                        464
BIHAR                              438
None                               200
UTTAR PRADESH                      196
ANDHRA PRADESH                     161
MAHARASHTRA                        158
JHARKHAND                          147
KARNATAKA                          142
CHHATTISGARH                       139
TAMIL NADU                         104
TELANGANA                           89
HARYANA                             81
PUNJAB                              53
RAJASTHAN                           49
KERALA                              45
GUJARAT                             43
CHANDIGARH                          42
UTTARAKHAND                         31
ASSAM                               30
JAMMU AND KASHMIR                   24
SIKKIM                              14
ARUNACHAL PRADESH                   14
HIMACHAL PRADESH   

In [ ]:
df_work['state'].apply(type).value_counts()


state
<class 'str'>         1371088
<class 'NoneType'>        200
Name: count, dtype: int64

In [ ]:
df_work['state'].isnull().sum()

np.int64(200)

In [ ]:
missing_state_rows = df_work[df_work['state'].isna()]
len(missing_state_rows)


200

In [ ]:
#View rows with missing state and district
missing_state_rows[['state', 'district']].head(10)


,state,district
1339,None,None
13475,None,None
31505,None,None
32517,None,None
37513,None,None
38463,None,None
45974,None,None
54696,None,None
65781,None,None
69432,None,None


> **Note:**  
> Standardization steps are commented out, so all original values (including `NaN` and `None`) were kept unchanged.

In [ ]:
# Replace missing state with "None"
#df_work.loc[df_work['state'].isna(), 'state'] = 'None'

# For those same rows where state was NoneType,
# also set district and block as "None"
#df_work.loc[df_work['state'] == 'None', ['district']] = 'None'


In [ ]:
#df_work[df_work['state'] == None][['state', 'district']].head(10)

,state,district


In [ ]:
#df_work['state'].isna().sum()

np.int64(0)

## District Check

In [ ]:
df_work[['district', 'district_id']].isna().sum()

district       9419
district_id    4610
dtype: int64

In [ ]:
#Count of unique combinations of district and district_id
df_work[['district', 'district_id']].drop_duplicates().shape[0]

34

In [ ]:
#View Unique combinations of district and district_id
df_work[['district', 'district_id']].value_counts(dropna=False)

district        district_id
Ganjam          354.0          98108
Baleswar        346.0          81910
Khordha         362.0          76244
Sambalpur       371.0          76241
Kalahandi       358.0          74474
Balangir        345.0          71167
Jajpur          356.0          68783
Bhadrak         348.0          67112
Cuttack         350.0          66109
Dhenkanal       352.0          56715
Mayurbhanj      365.0          54039
Puri            369.0          51967
Nayagarh        367.0          51735
Kendrapara      360.0          47184
Kendujhar       361.0          45236
Bargarh         347.0          44352
Jagatsinghapur  355.0          43775
Sundargarh      373.0          43226
Angul           344.0          36342
Subarnapur      372.0          25939
Nuapada         368.0          25648
Kandhamal       359.0          22882
Nabarangpur     366.0          22814
Koraput         363.0          21444
Gajapati        353.0          20707
Boudh           349.0          18198
Rayagada  

**Insights (Require further verification)**

- Records from non-Odisha states have district_id = 0 or NaN
- Several Odisha districts are mapped to multiple other states
- Some entries show impossible combinations (e.g., Odisha districts under Haryana or Delhi)

In [ ]:
#View max rows
pd.set_option('display.max_rows', 200)
df_work[['district_id', 'district', 'state']].drop_duplicates().sort_values('district_id')


,district_id,district,state
76130,0.0,None,GOA
75884,0.0,None,SIKKIM
751,0.0,None,ANDHRA PRADESH
865,0.0,None,ODISHA
970,0.0,None,MADHYA PRADESH
1069,0.0,None,DELHI
1097,0.0,None,CHHATTISGARH
1339,0.0,None,None
1739,0.0,None,RAJASTHAN
2894,0.0,None,PUDUCHERRY


**Mapping District Ids from admin demographic hierarchy file**

In [28]:
#load csv
district_ref = pd.read_csv("/Users/ghazalhashmi/Library/CloudStorage/Box-Box/Ghazal/Data/Raw/janasunani-mappings/m_demographic_hierarchy_values.csv")


In [ ]:
#keep only district level enteries
district_ref = district_ref[
    (district_ref['intDemoParentValueId'] == 1) &
    (district_ref['intSubLevelId'] == 1)
][['intDemoHierarchyValueId', 'vchDemoHierarchyValue']]

district_ref.rename(columns={
    'intDemoHierarchyValueId': 'district_id_ref',
    'vchDemoHierarchyValue': 'district_ref'
}, inplace=True)


In [ ]:
#prepare working data for matching
df_check = df_work[['district', 'district_id']].drop_duplicates().copy()
df_check['district'] = df_check['district'].astype(str).str.strip().str.lower()


In [ ]:
#merge to check consistency
merged_district = df_check.merge(
    district_ref,
    left_on=['district_id'],
    right_on=['district_id_ref'],
    how='left',
    indicator=True
)

# Flag matches
merged_district['match_status'] = merged_district['_merge'].map({
    'both': 'Matched',
    'left_only': 'Not Matched'
})




In [ ]:
#print summary
print("District ID-Name Consistency Check:")
print(merged_district['match_status'].value_counts())

# Show unmatched departments
unmatched_district = merged_district[merged_district['match_status'] == 'Not Matched'][['district', 'district_id']]
print(f"\nTotal unmatched districts {len(unmatched_district)}")
display(unmatched_district)


District ID-Name Consistency Check:
match_status
Matched        30
Not Matched     4
Name: count, dtype: int64

Total unmatched districts 4


,district,district_id
4,none,NaN
23,none,0.0
32,none,1.0
33,none,3089.0


> **Note:**  
> Standardization steps are commented out, so all original values (including `NaN` and `None`) were kept unchanged.

In [ ]:
# Replace invalid district IDs where name is 'none' or missing
#df_work.loc[df_work['district'].str.lower().eq('none'), 'district_id'] = 0

# Also fix if district_id is NaN but name is none
#df_work['district_id'] = df_work['district_id'].fillna(0)

# Ensure correct types
#df_work['district_id'] = df_work['district_id'].astype(int)


In [ ]:
# Replace invalid IDs and names
#df_work.loc[
    #df_work['district_id'].isin([0, 1, 3089]) | df_work['district'].isin(['NaN', 'None', '']),
    ['district', 'district_id']
] = ['None', 0]

# Verify the update
#print(df_work[['district', 'district_id']].value_counts(dropna=False))

district        district_id
Ganjam          354            98108
Baleswar        346            81910
Khordha         362            76244
Sambalpur       371            76241
Kalahandi       358            74474
Balangir        345            71167
Jajpur          356            68783
Bhadrak         348            67112
Cuttack         350            66109
Dhenkanal       352            56715
Mayurbhanj      365            54039
Puri            369            51967
Nayagarh        367            51735
Kendrapara      360            47184
Kendujhar       361            45236
Bargarh         347            44352
Jagatsinghapur  355            43775
Sundargarh      373            43226
Angul           344            36342
Subarnapur      372            25939
Nuapada         368            25648
Kandhamal       359            22882
Nabarangpur     366            22814
Koraput         363            21444
Gajapati        353            20707
Boudh           349            18198
Rayagada  

In [ ]:
#df_work.groupby('district')['district_id'].nunique().value_counts()


district_id
1    31
Name: count, dtype: int64

In [ ]:
#Convert float to Int64
df_work['district_id'] = df_work['district_id'].astype('Int64')


In [ ]:
df_work['district_id'].dtype


Int64Dtype()

In [ ]:
#df_work['district_id'].isna().sum()


np.int64(0)

## Category Column Check

In [ ]:
df_work[['category', 'category_id']].isna().sum()

category       231452
category_id         0
dtype: int64

In [ ]:
df_work[['category', 'category_id']].value_counts(dropna=False)

category                     category_id
Housing                      52             304678
NaN                          0              231452
Social Welfare               7              215132
General                      4              141677
Miscellaneous                55              92184
Infrastructure               10              80928
Police Case                  6               60127
Land Matters                 54              50231
Service Matters              11              46927
Financial Assistance         51              19817
Energy                       14              19404
Water Supply                 39              18938
Health Care                  8               15558
Public Utility               56              15025
Education                    9               14503
Pension/Retirement Benefits  3                8880
Agriculture & Farming        12               7438
School & College             34               6785
Irrigation                   58          

In [ ]:
# Same category name, multiple IDs
dup_name_ids = (
    df_work[['category', 'category_id']]
    .dropna()
    .drop_duplicates()
    .groupby('category')['category_id']
    .nunique()
)
dup_name_ids = dup_name_ids[dup_name_ids > 1]
display(dup_name_ids)


category
COVID-19        2
Education       2
General         2
Health Care     2
Water Supply    2
Name: category_id, dtype: int64

**Insights (Require futher verification)**

- Same category appears with different category_ids
- Several subcategories are linked to multiple category_ids


In [ ]:
# Get categories that have multiple IDs
dup_categories = dup_name_ids.index.tolist()

# Filter only those categories
dup_subcats = (
    df_work[df_work['category'].isin(dup_categories)][
        ['category', 'category_id', 'subcategory', 'subcategory_id']
    ]
    .drop_duplicates()
    .sort_values(['category', 'category_id', 'subcategory'])
)

display(dup_subcats)


,category,category_id,subcategory,subcategory_id
2002,COVID-19,47,Covid Treatment Assistance,216
5031,COVID-19,47,Others,218
23028,COVID-19,47,Quarantine Centre/Hospital Issues,217
503000,COVID-19,48,Others,215
47,Education,9,Higher Education,12
1217,Education,9,Higher Secondary Education,337
507062,Education,9,Others,38
498,Education,9,Primary/Secondary Education,43
86,Education,9,Scholarship/study Loan,54
2262,Education,9,Technical Education/Professional Education,61


**Mapping Category Ids from admin category file**

In [ ]:
# Load the reference mapping (official admin category master)
cat_ref = pd.read_csv("/Users/ghazalhashmi/Library/CloudStorage/Box-Box/Ghazal/Data/Raw/janasunani-mappings/m_admin_category.csv")

In [ ]:
# Keep only relevant columns and clean text
cat_ref = cat_ref[['intCategoryId', 'vchCategory']]
cat_ref.columns = ['category_id_ref', 'category_ref']
cat_ref['category_ref'] = cat_ref['category_ref'].str.strip().str.lower()

In [ ]:
# Clean df_work columns for comparison
df_check = df_work[['category', 'category_id']].copy()
df_check['category'] = df_check['category'].astype(str).str.strip().str.lower()

In [ ]:
# Merge to check mismatches
merged_cat = df_check.merge(
    cat_ref,
    left_on='category_id',
    right_on='category_id_ref',
    how='left',
    indicator=True
)

In [ ]:
# Compare expected vs. actual category name
merged_cat['match_status'] = merged_cat.apply(
    lambda x: 'Matched' if x['category'] == x['category_ref'] else 'Mismatch', axis=1
)

In [ ]:
# Summary
print("\nCategory ID-name consistency check:")
print(merged_cat['match_status'].value_counts())


Category ID-name consistency check:
match_status
Matched     1139836
Mismatch     231452
Name: count, dtype: int64


In [ ]:

# Display mismatched records
mismatched = merged_cat[merged_cat['match_status'] == 'Mismatch'][['category', 'category_id', 'category_ref']]
print(f"\nTotal mismatched category entries: {len(mismatched)}")
display(mismatched)


Total mismatched category entries: 231452


,category,category_id,category_ref
9,none,0,NaN
11,none,0,NaN
12,none,0,NaN
13,none,0,NaN
14,none,0,NaN
...,...,...,...
1371280,none,0,NaN
1371282,none,0,NaN
1371285,none,0,NaN
1371286,none,0,NaN


> **Note:**  
> Standardization steps are commented out, so all original values (including `NaN` and `None`) were kept unchanged.

In [ ]:
# Replace all invalid or missing category values
#df_work.loc[df_work['category'].isna() | (df_work['category'] == 'None'), 'category'] = 'None'
#df_work.loc[df_work['category_id'].isna() | (df_work['category_id'] == 0), 'category_id'] = 0


In [ ]:
#df_work[['category', 'category_id']].value_counts(dropna=False)


category                     category_id
Housing                      52             304678
None                         0              231452
Social Welfare               7              215132
General                      4              141677
Miscellaneous                55              92184
Infrastructure               10              80928
Police Case                  6               60127
Land Matters                 54              50231
Service Matters              11              46927
Financial Assistance         51              19817
Energy                       14              19404
Water Supply                 39              18938
Health Care                  8               15558
Public Utility               56              15025
Education                    9               14503
Pension/Retirement Benefits  3                8880
Agriculture & Farming        12               7438
School & College             34               6785
Irrigation                   58          

In [ ]:
#df_work[['category', 'category_id']].isna().sum()

category       0
category_id    0
dtype: int64

## Subcategory Check

In [ ]:
df_work[['subcategory', 'subcategory_id']].isna().sum()

subcategory       206924
subcategory_id         0
dtype: int64

In [ ]:
df_work[['subcategory', 'subcategory_id']].value_counts(dropna=False)

subcategory                 subcategory_id
Rural Housing               240               302378
NaN                         0                 206908
IAY/MKY/BPGY/PMAY           13                128321
Others                      38                123563
Miscellaneous/Others        266                63116
                                               ...  
Higher Education            80                     1
IAY/MKY/BPGY/BAY            81                     1
Police Cases                373                    1
Social Welfare Scheme       126                    1
Health Care Infrastructure  152                    1
Name: count, Length: 268, dtype: int64

In [30]:
# Temporarily expand display limit
pd.set_option('display.max_rows', None)

# Show full frequency table
subcategory_counts = df_work[['subcategory', 'subcategory_id']].value_counts(dropna=False)
display(subcategory_counts)


subcategory                                       subcategory_id
Rural Housing                                     240               302378
NaN                                               0                 206908
IAY/MKY/BPGY/PMAY                                 13                128321
Others                                            38                123563
Miscellaneous/Others                              266                63116
Police Cases                                      41                 47456
Road                                              51                 26267
Excise Matters                                    2                  25893
Service Matters                                   56                 24781
Land Allotment/Settlement/RoR                     247                21854
Allegation/Corruption                             252                21200
Ration Card Issue                                 165                19730
Religious Institution              

In [ ]:
# Subcategories with multiple IDs
multi_ids = (
    df_work[['subcategory', 'subcategory_id']]
    .dropna()
    .drop_duplicates()
    .groupby('subcategory')['subcategory_id']
    .nunique()
)
multi_ids = multi_ids[multi_ids > 1]
print(f"Subcategories linked to multiple IDs: {len(multi_ids)}")
display(multi_ids)


Subcategories linked to multiple IDs: 20


subcategory
Development of Tourist Places       2
Encroachment                        2
Environment Pollution/Protection    2
Family Problem                      2
Financial Assistance                2
Higher Education                    2
Insurance Matters                   2
Missing Person                      2
Natural Calamities/ Relief          3
Other Pension                       2
Others                              5
Police Cases                        2
Property Dispute                    2
Protection of Life and Property     2
Public Utility Problem              2
Rehabilitation Assistance Scheme    2
Rehabilitation of the Distressed    2
Social Welfare Scheme               2
Torture/Harassment                  2
Various Demands                     2
Name: subcategory_id, dtype: int64

In [ ]:
#Check duplicate subcategory names with different IDs
dup_subs = (
    df_work[['subcategory', 'subcategory_id']]
    .drop_duplicates()
    .groupby('subcategory')['subcategory_id']
    .apply(lambda x: ', '.join(map(str, sorted(x))))
    .reset_index()
)
dup_subs = dup_subs[dup_subs['subcategory_id'].str.contains(',')]
print(f"Total duplicated subcategory names: {len(dup_subs)}")
display(dup_subs)


Total duplicated subcategory names: 20


,subcategory,subcategory_id
46,Development of Tourist Places,"284, 363"
58,Encroachment,"205, 245"
61,Environment Pollution/Protection,"1, 289"
67,Family Problem,"4, 72"
70,Financial Assistance,"138, 162"
86,Higher Education,"12, 80"
106,Insurance Matters,"16, 234"
137,Missing Person,"28, 370"
143,Natural Calamities/ Relief,"31, 99, 235"
157,Other Pension,"37, 307"


**Insight (require further check)**
- Same subcategory appears under multiple categories

In [ ]:
#Focus on specific known duplicate subcategories
dup_subs_list = [
    'Development of Tourist Places', 'Encroachment', 'Environment Pollution/Protection',
    'Family Problem', 'Financial Assistance', 'Higher Education', 'Insurance Matters',
    'Missing Person', 'Natural Calamities/ Relief', 'Other Pension', 'Others', 'Police Cases',
    'Property Dispute', 'Protection of Life and Property', 'Public Utility Problem',
    'Rehabilitation Assistance Scheme', 'Rehabilitation of the Distressed',
    'Social Welfare Scheme', 'Torture/Harassment', 'Various Demands'
]

df_work[df_work['subcategory'].isin(dup_subs_list)][
    ['category_id', 'category', 'subcategory', 'subcategory_id']
].drop_duplicates().sort_values(['subcategory', 'category'])


,category_id,category,subcategory,subcategory_id
1215,56,Public Utility,Development of Tourist Places,284
241,33,Tourism,Development of Tourist Places,363
4014,4,General,Encroachment,205
117,54,Land Matters,Encroachment,245
225010,6,Police Case,Encroachment,205
858,1,Environment,Environment Pollution/Protection,1
34311,56,Public Utility,Environment Pollution/Protection,289
1029,4,General,Family Problem,4
878003,20,General,Family Problem,72
324,12,Agriculture & Farming,Financial Assistance,162


**Mapping Subcategory Ids from admin Subcategory file**

In [ ]:
# Load the reference (admin subcategory master)
sub_ref = pd.read_csv("/Users/ghazalhashmi/Library/CloudStorage/Box-Box/Ghazal/Data/Raw/janasunani-mappings/m_admin_subcategory.csv")


In [ ]:
# Keep all (ignore deletion or active flags)
sub_ref = sub_ref[['intSubCategoryId', 'intCategoryId', 'vchSubCategory']]
sub_ref['vchSubCategory'] = sub_ref['vchSubCategory'].astype(str).str.strip().str.lower()


In [ ]:
# Rename columns for clarity
sub_ref = sub_ref.rename(columns={
    'intSubCategoryId': 'subcategory_id_ref',
    'vchSubCategory': 'subcategory_ref',
    'intCategoryId': 'category_id_ref'
})


In [ ]:
# Prepare your working dataframe
df_check = df_work[['category', 'subcategory', 'subcategory_id']].drop_duplicates().copy()
df_check['subcategory'] = df_check['subcategory'].astype(str).str.strip().str.lower()

# Merge to check matches
merged_sub = df_check.merge(
    sub_ref[['subcategory_id_ref', 'subcategory_ref']],
    left_on='subcategory_id',
    right_on='subcategory_id_ref',
    how='left',
    indicator=True
)

# Flag matched/unmatched
merged_sub['match_status'] = merged_sub['_merge'].map({
    'both': 'Matched',
    'left_only': 'Not Matched'
})

print("\nSubcategory ID-name consistency check:")
print(merged_sub['match_status'].value_counts())


Subcategory ID-name consistency check:
match_status
Matched        321
Not Matched      2
Name: count, dtype: int64


In [ ]:
# Show only missing or inconsistent mappings
unmatched_subs = merged_sub[merged_sub['match_status'] == 'Not Matched'][[
    'category', 'subcategory', 'subcategory_id','subcategory_id_ref', 'subcategory_ref'
]]

print(f"\nTotal unmatched subcategories: {len(unmatched_subs)}")
display(unmatched_subs)


Total unmatched subcategories: 2


,category,subcategory,subcategory_id,subcategory_id_ref,subcategory_ref
7,None,none,0,NaN,NaN
272,Police Case,none,25,NaN,NaN


In [ ]:
# Count how many records have subcategory_id 0 or 25
df_work['subcategory_id'].value_counts().loc[[0,25]]


subcategory_id
0     206908
25        10
Name: count, dtype: int64

> **Note:**  
> Standardization steps are commented out, so all original values (including `NaN` and `None`) were kept unchanged.

In [ ]:
# Replace subcategory name as 'none' where subcategory_id = 0
#df_work.loc[df_work['subcategory_id'] == 0, 'subcategory'] = 'None'

In [ ]:
#df_work['subcategory'].isna().sum() 
#df_work['subcategory_id'].isna().sum()


np.int64(0)

In [ ]:
missing_subcats = (
    df_work[df_work['subcategory'].isna()]
    .groupby(['category', 'category_id', 'subcategory', 'subcategory_id'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
display(missing_subcats)


,category,category_id,subcategory,subcategory_id,count
3,NaN,0,NaN,0,206908
2,Police Case,6,NaN,25,10
1,Land Matters,54,NaN,23,4
0,Land Matters,54,NaN,22,2


In [ ]:
#Check subcategory names in the admin file
sub_ref= sub_ref[sub_ref["subcategory_id_ref"].isin([22, 23, 25])]
display(sub_ref[["subcategory_id_ref", "subcategory_ref", "category_id_ref"]])


,subcategory_id_ref,subcategory_ref,category_id_ref
21,22,land allotment settlement,4
22,23,land dispute,4


In [ ]:
# Update subcategory names for IDs 22 and 23
df_work.loc[df_work["subcategory_id"] == 22, "subcategory"] = "Land allotment settlement"
df_work.loc[df_work["subcategory_id"] == 23, "subcategory"] = "Land Dispute"


In [ ]:
#updated value counts
df_work[['subcategory', 'subcategory_id']].value_counts(dropna=False)


subcategory                                       subcategory_id
Rural Housing                                     240               302378
NaN                                               0                 206908
IAY/MKY/BPGY/PMAY                                 13                128321
Others                                            38                123563
Miscellaneous/Others                              266                63116
Police Cases                                      41                 47456
Road                                              51                 26267
Excise Matters                                    2                  25893
Service Matters                                   56                 24781
Land Allotment/Settlement/RoR                     247                21854
Allegation/Corruption                             252                21200
Ration Card Issue                                 165                19730
Religious Institution              

## Office Column Check

In [ ]:
df_work[['office', 'office_id']].isna().sum()

office       59450
office_id        0
dtype: int64

In [ ]:
df_work[['office', 'office_id']].value_counts(dropna=False)

office                    office_id
Collector                 6            693691
Departments               5            280305
Office of Chief Minister  1            217253
NaN                       0             59450
Superintendent of Police  7             42757
Chief Secretary           3             33037
DG & IG Police            4             28911
Governor                  2             15884
Name: count, dtype: int64

**Mapping Office Ids from admin office file**

In [ ]:
# Load reference file
office_ref = pd.read_csv("/Users/ghazalhashmi/Library/CloudStorage/Box-Box/Ghazal/Data/Raw/janasunani-mappings/m_admin_offices.csv")

In [ ]:
# Clean reference data
office_ref = office_ref[['intOfficeId', 'vchOfficeName']]
office_ref.columns = ['office_id_ref', 'office_ref']
office_ref['office_ref'] = office_ref['office_ref'].astype(str).str.strip().str.lower()


In [ ]:
# Clean df_work office data
df_check = df_work[['office', 'office_id']].drop_duplicates().copy()
df_check['office'] = df_check['office'].astype(str).str.strip().str.lower()

In [ ]:
# Merge to compare
merged_offices = df_check.merge(
    office_ref,
    left_on=['office_id'],
    right_on=['office_id_ref'],
    how='left',
    indicator=True
)

# Flag match status
merged_offices['match_status'] = merged_offices['_merge'].map({
    'both': 'Matched',
    'left_only': 'Not Matched'
})

# Summary
print("Office ID-name consistency check:")
print(merged_offices['match_status'].value_counts())


Office ID-name consistency check:
match_status
Matched        7
Not Matched    1
Name: count, dtype: int64


In [ ]:
# Show mismatches
unmatched = merged_offices[merged_offices['match_status'] == 'Not Matched'][
    ['office', 'office_id']
]
print(f"\nTotal unmatched offices: {len(unmatched)}")
display(unmatched)


Total unmatched offices: 1


,office,office_id
0,none,0


> **Note:**  
> Standardization steps are commented out, so all original values (including `NaN` and `None`) were kept unchanged.

In [ ]:
# Standardize 'none' office entries
#df_work.loc[df_work['office_id'] == 0, 'office'] = 'None'

# Reconfirm unique mapping
#print(df_work[['office', 'office_id']].value_counts(dropna=False))


office                    office_id
Collector                 6            693691
Departments               5            280305
Office of Chief Minister  1            217253
None                      0             59450
Superintendent of Police  7             42757
Chief Secretary           3             33037
DG & IG Police            4             28911
Governor                  2             15884
Name: count, dtype: int64


## Dept Column Check

In [ ]:
df_work[['dept', 'dept_id']].isna().sum()

dept       210450
dept_id         0
dtype: int64

In [ ]:
df_work[['dept', 'dept_id']].value_counts(dropna=False)

dept                                                        dept_id
Panchayati Raj & Drinking Water                             21         529708
NaN                                                         0          210445
Home department                                             14          93335
Revenue & Disaster Management                               26          88410
General Administration & Public Grievance                   11          71837
Housing & Urban Development                                 15          56348
Social Security & Empowerment of Persons with Disabilities  40          45339
School & Mass Education                                     29          40163
Health & Family Welfare                                     12          33108
Food Supplies & Consumer Welfare                            9           29498
Agriculture & Farmers' Empowerment                          1           20734
Energy                                                      5           18

**Mapping Dept Ids from admin hierarchy file**

In [ ]:
# Load the department hierarchy reference file
dept_ref = pd.read_csv("/Users/ghazalhashmi/Library/CloudStorage/Box-Box/Ghazal/Data/Raw/janasunani-mappings/m_admin_hierarchy_value.csv")

In [ ]:
# Keep only department-level entries (intAdminLabelId == 1)
dept_ref = dept_ref[dept_ref['intAdminLabelId'] == 1][[
    'intAdminHierarchyValueId', 'vchAdminHierarchyValue'
]].copy()

In [ ]:
# Clean and rename columns
dept_ref = dept_ref.rename(columns={
    'intAdminHierarchyValueId': 'dept_id_ref',
    'vchAdminHierarchyValue': 'dept_ref'
})
dept_ref['dept_ref'] = dept_ref['dept_ref'].astype(str).str.strip().str.lower()

In [ ]:
#Prepare Working Data for Matching
df_check = df_work[['dept', 'dept_id']].drop_duplicates().copy()
df_check['dept'] = df_check['dept'].astype(str).str.strip().str.lower()


In [ ]:
#Merge to Check Consistency
merged_dept = df_check.merge(
    dept_ref,
    left_on=['dept_id'],
    right_on=['dept_id_ref'],
    how='left',
    indicator=True
)

# Flag matches
merged_dept['match_status'] = merged_dept['_merge'].map({
    'both': 'Matched',
    'left_only': 'Not Matched'
})


In [ ]:
#print summary
print("Department ID-Name Consistency Check:")
print(merged_dept['match_status'].value_counts())

# Show unmatched departments
unmatched_dept = merged_dept[merged_dept['match_status'] == 'Not Matched'][['dept', 'dept_id']]
print(f"\nTotal unmatched departments: {len(unmatched_dept)}")
display(unmatched_dept)


Department ID-Name Consistency Check:
match_status
Matched        40
Not Matched     6
Name: count, dtype: int64

Total unmatched departments: 6


,dept,dept_id
6,none,0
41,none,98
42,none,2380
43,none,103
44,none,94
45,none,112


In [ ]:
#invalid_dept_ids = [94, 98, 103, 112, 2380]

#df_work.loc[df_work['dept_id'].isin(invalid_dept_ids) | df_work['dept'].isna(), ['dept', 'dept_id']] = ['None', 0]


In [ ]:
#df_work[['dept', 'dept_id']].value_counts(dropna=False)


dept                                                        dept_id
Panchayati Raj & Drinking Water                             21         529708
None                                                        0          210450
Home department                                             14          93335
Revenue & Disaster Management                               26          88410
General Administration & Public Grievance                   11          71837
Housing & Urban Development                                 15          56348
Social Security & Empowerment of Persons with Disabilities  40          45339
School & Mass Education                                     29          40163
Health & Family Welfare                                     12          33108
Food Supplies & Consumer Welfare                            9           29498
Agriculture & Farmers' Empowerment                          1           20734
Energy                                                      5           18

## Received_by Column Check

In [ ]:
df_work['received_by'].value_counts(dropna=False)


received_by
CM Grievance Cell                       216844
Secretary , Panchayati Raj & DW         114882
Chief Minister Office                    58944
Collector, Ganjam                        57550
Collector, Kalahandi                     45478
Collector, Baleswar                      44914
Collector, Balangir                      41116
Collector, Mayurbhanj                    36503
Chief Secretary Odisha                   33037
Collector, Bhadrak                       29850
Collector, Sundargarh                    29557
Collector, Dhenkanal                     29316
DGP, Odisha Police HQ                    28911
Collector, Jajpur                        27638
Collector, Sambalpur                     27371
Collector, Jagatsinghpur                 24910
Collector, Cuttack                       24035
Collector, Bargarh                       23308
Collector, Kendujhar                     23054
Collector, Puri                          22239
Collector, Khordha                       21586
C

> **Note:**  
> Standardization steps are commented out, so all original values (including `NaN` and `None`) were kept unchanged.

In [ ]:
#df_work['received_by'] = df_work['received_by'].fillna('None')


In [ ]:
#df_work.isnull().mean().sort_values(ascending=False) * 100

document_url         23.641132
id                    0.000000
dept                  0.000000
hour                  0.000000
day_of_week           0.000000
day                   0.000000
month_name            0.000000
month                 0.000000
year                  0.000000
document_attached     0.000000
ticket_length         0.000000
petitioner_gender     0.000000
state                 0.000000
subcategory           0.000000
subcategory_id        0.000000
dept_id               0.000000
ticket_no             0.000000
category              0.000000
category_id           0.000000
created_on            0.000000
status                0.000000
disability            0.000000
mode                  0.000000
district              0.000000
district_id           0.000000
received_by           0.000000
office                0.000000
office_id             0.000000
date                  0.000000
dtype: float64